# Self Concept Literature - Network Analysis

This notebook builds publication-ready static questionnaire validity/correlation network figures from `tier1_validity_extraction.xlsx`.

The default node universe is defined by questionnaires/scales appearing in `focal_questionnaire`. A small, explicit exception set is added from `related_questionnaire` because these scales are theoretically relevant to the manuscript figure:

- `(PSCQ): Past Self-Continuity Questionnaire`
- `(ES): Essentialist Beliefs about Human Kinds Scale`
- `(IPTheory - S): Implicit person theory measure - specific abilities`

Rows where `related_questionnaire` points to other outside instruments are excluded from the graph and saved for QA.


## 1. Load the Workbook and Validate Columns

The workbook is loaded with pandas. Column names are normalized only for minor spelling/case/spacing differences. Major missing columns stop the notebook with a clear error.


In [1]:
from pathlib import Path
from collections import Counter
import math
import re
import textwrap

import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

GRAPH_TITLE = "Self concept literature - network analysis"
INPUT_FILENAME = "tier1_validity_extraction.xlsx"
PREFERRED_SHEET = "validity_edges"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / INPUT_FILENAME
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_GRAPH_PATHS = {
    "png": OUTPUT_FIGURES_DIR / "self questionnaires mapping network.png",
    "pdf": OUTPUT_FIGURES_DIR / "self questionnaires mapping network.pdf",
    "svg": OUTPUT_FIGURES_DIR / "self questionnaires mapping network.svg",
}
SELF_CONCEPT_CLUSTER_PATHS = {
    "png": OUTPUT_FIGURES_DIR / "self_concept_cluster_network.png",
    "pdf": OUTPUT_FIGURES_DIR / "self_concept_cluster_network.pdf",
    "svg": OUTPUT_FIGURES_DIR / "self_concept_cluster_network.svg",
}
ESSENTIALISM_CLUSTER_PATHS = {
    "png": OUTPUT_FIGURES_DIR / "essentialism_cluster_network.png",
    "pdf": OUTPUT_FIGURES_DIR / "essentialism_cluster_network.pdf",
    "svg": OUTPUT_FIGURES_DIR / "essentialism_cluster_network.svg",
}

CLEANED_EDGES_PATH = OUTPUT_TABLES_DIR / "cleaned_network_edges.csv"
NODE_LIST_PATH = OUTPUT_TABLES_DIR / "network_nodes.csv"
NODE_LABEL_KEY_PATH = OUTPUT_TABLES_DIR / "network_node_label_key.csv"
EXCLUDED_ROWS_QA_PATH = OUTPUT_TABLES_DIR / "excluded_rows_QA.csv"
EXCLUDED_NON_FOCAL_PATH = OUTPUT_TABLES_DIR / "excluded_non_focal_related_questionnaires.csv"

ADDITIONAL_RELATED_QUESTIONNAIRES = [
    "(PSCQ): Past Self-Continuity Questionnaire",
    "(ES): Essentialist Beliefs about Human Kinds Scale",
    "(IPTheory - S): Implicit person theory measure - specific abilities",
]

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected input file not found: {DATA_PATH}")

xlsx = pd.ExcelFile(DATA_PATH)
SHEET_NAME = PREFERRED_SHEET if PREFERRED_SHEET in xlsx.sheet_names else xlsx.sheet_names[0]
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print(f"Workbook: {DATA_PATH}")
print(f"Available sheets: {xlsx.sheet_names}")
print(f"Selected sheet: {SHEET_NAME}")
print("Actual column names:")
for i, column in enumerate(df_raw.columns, start=1):
    print(f"  {i}. {column!r}")

REQUIRED_COLUMNS = [
    "focal_questionnaire",
    "related_questionnaire",
    "relation_value",
    "significance",
    "relation_source_type",
]

def canonicalize_column_name(column):
    """Normalize column names for minor case/spacing/punctuation differences."""
    text = str(column).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

actual_by_canonical = {}
duplicate_canonical_names = []
for column in df_raw.columns:
    canonical = canonicalize_column_name(column)
    if canonical in actual_by_canonical:
        duplicate_canonical_names.append((canonical, actual_by_canonical[canonical], column))
    else:
        actual_by_canonical[canonical] = column

if duplicate_canonical_names:
    print("Warning: duplicate columns after normalization:")
    for canonical, first, second in duplicate_canonical_names:
        print(f"  {canonical!r}: {first!r} and {second!r}")

missing_columns = [column for column in REQUIRED_COLUMNS if canonicalize_column_name(column) not in actual_by_canonical]
if missing_columns:
    raise ValueError(
        "Missing required columns after minor normalization: "
        + ", ".join(missing_columns)
        + "\nActual columns were: "
        + ", ".join(map(str, df_raw.columns))
    )

column_mapping = {
    actual_by_canonical[canonicalize_column_name(expected)]: expected
    for expected in REQUIRED_COLUMNS
}

df = df_raw.rename(columns=column_mapping).copy()

print("\nColumn mapping used:")
for actual, expected in column_mapping.items():
    print(f"  {actual!r} -> {expected!r}")


Workbook: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/data/tier1_validity_extraction.xlsx
Available sheets: ['validity_edges', 'source_status']
Selected sheet: validity_edges
Actual column names:
  1. 'focal_questionnaire'
  2. 'Tier'
  3. 'is_adapted_version'
  4. 'adapted_from'
  5. 'related_questionnaire'
  6. 'is_related_questionnaire_a_subscale'
  7. 'relation_value'
  8. 'Significance'
  9. 'relation_source_type'
  10. 'article_reference'
  11. 'notes'

Column mapping used:
  'focal_questionnaire' -> 'focal_questionnaire'
  'related_questionnaire' -> 'related_questionnaire'
  'relation_value' -> 'relation_value'
  'Significance' -> 'significance'
  'relation_source_type' -> 'relation_source_type'


## 2. Clean Questionnaire Names and Parse Relation Values

Questionnaire labels are cleaned for matching while preserving readable display text. Relation values are parsed from common forms such as `.45`, `-.32`, `r = .45`, and `r=.45`.


In [2]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def questionnaire_key(value):
    """Stable matching key for questionnaire names."""
    return clean_text(value).casefold()


def parse_relation_value(value):
    """Parse a numeric correlation-like value from common validity/correlation text."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)

    text = clean_text(value)
    if not text:
        return pd.NA

    text = (
        text.replace("−", "-")
        .replace("–", "-")
        .replace("—", "-")
        .replace(",", ".")
    )

    labelled_match = re.search(
        r"(?:^|[\s,;(])(?:r|sp|sr|semipartial|semi-partial|partial)\s*=?\s*([-+]?(?:\d*\.\d+|\d+\.?\d*))",
        text,
        flags=re.IGNORECASE,
    )
    if labelled_match:
        candidate = labelled_match.group(1)
    else:
        fallback_match = re.search(r"[-+]?(?:\d*\.\d+|\d+\.?\d*)", text)
        if not fallback_match:
            return pd.NA
        candidate = fallback_match.group(0)

    try:
        return float(candidate)
    except ValueError:
        return pd.NA


def split_questionnaire_label(label):
    """Split labels like '(SCC): Self-Concept Clarity Scale' into acronym and name."""
    text = clean_text(label).replace("*", "")
    match = re.match(r"^(\([^:]+\))\s*:\s*(.+)$", text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    match = re.match(r"^([^:]{2,40})\s*:\s*(.+)$", text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return "", text


def short_node_label(label):
    acronym, full_name = split_questionnaire_label(label)
    text = acronym.strip("()") if acronym else full_name
    text = text.replace(" - ", "-")
    text = re.sub(r"\s*\(\s*", "-", text).replace(")", "")
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"^IPTheory", "IPT", text)
    return text


def plot_node_label(label, width=28):
    """Full node label for the figure, preserving acronym and questionnaire name."""
    acronym, full_name = split_questionnaire_label(label)
    if acronym and full_name:
        return f"{acronym}:\n{textwrap.fill(full_name, width=width)}"
    return textwrap.fill(clean_text(label).replace("*", ""), width=width)


def format_relation_number(value):
    text = f"{float(value):.2f}"
    if text.startswith("-0"):
        return "-" + text[2:]
    if text.startswith("0"):
        return text[1:]
    return text


def format_edge_label(value, source_type):
    source_type = clean_text(source_type).casefold()
    number = format_relation_number(value)
    if source_type == "regression_semipartial correlation":
        return f"sp={number}"
    return f"r={number}"


def edge_color(value):
    # Okabe-Ito vermillion and blue: colorblind-aware while preserving red/blue valence.
    if float(value) > 0:
        return "#D55E00"
    if float(value) < 0:
        return "#0072B2"
    return "#666666"


def edge_linestyle(source_type):
    source_type = clean_text(source_type).casefold()
    if source_type == "convergent_validity":
        return "--"
    if source_type == "regression_semipartial correlation":
        return ":"
    return "-"


def show_table(table, title=None):
    if title:
        print(title)
    try:
        display(table)
    except NameError:
        print(table.to_string(index=False))


df["focal_clean"] = df["focal_questionnaire"].map(clean_text)
df["related_clean"] = df["related_questionnaire"].map(clean_text)
df["focal_key"] = df["focal_clean"].map(questionnaire_key)
df["related_key"] = df["related_clean"].map(questionnaire_key)
df["significance_clean"] = df["significance"].map(lambda value: clean_text(value).casefold())
df["source_type_clean"] = df["relation_source_type"].map(lambda value: clean_text(value).casefold())
df["source_type_clean"] = df["source_type_clean"].str.replace(r"\s+", " ", regex=True)
df["relation_value_numeric"] = pd.to_numeric(df["relation_value"].map(parse_relation_value), errors="coerce")


## 3. Define Nodes, Filter Edges, and Build QA Tables

All focal questionnaires become nodes. The three manually specified related-only questionnaires are also allowed into the analysis node set. Edges are included only when the focal questionnaire is focal, the related questionnaire is in the analysis node set, and all significance/source/value filters pass.


In [3]:
ALLOWED_SOURCE_TYPES = {
    "correlation_only",
    "convergent_validity",
    "regression_semipartial correlation",
}

focal_display_by_key = (
    df.loc[df["focal_key"] != "", ["focal_key", "focal_clean"]]
    .drop_duplicates("focal_key")
    .set_index("focal_key")["focal_clean"]
    .to_dict()
)
focal_keys = set(focal_display_by_key)

requested_related_by_key = {questionnaire_key(label): clean_text(label) for label in ADDITIONAL_RELATED_QUESTIONNAIRES}
related_display_by_key = (
    df.loc[df["related_key"].isin(requested_related_by_key), ["related_key", "related_clean"]]
    .drop_duplicates("related_key")
    .set_index("related_key")["related_clean"]
    .to_dict()
)
missing_requested_related = [
    label for key, label in requested_related_by_key.items() if key not in related_display_by_key and key not in focal_display_by_key
]
if missing_requested_related:
    raise ValueError(
        "The following manually requested related questionnaires were not found exactly in related_questionnaire: "
        + "; ".join(missing_requested_related)
    )

analysis_display_by_key = dict(focal_display_by_key)
for key, label in related_display_by_key.items():
    analysis_display_by_key.setdefault(key, label)
analysis_keys = set(analysis_display_by_key)
manual_related_only_keys = set(related_display_by_key) - focal_keys

nodes_df = pd.DataFrame(
    [
        {
            "questionnaire_key": key,
            "questionnaire_label": label,
            "node_origin": "focal_questionnaire" if key in focal_keys else "manual_related_inclusion",
        }
        for key, label in sorted(analysis_display_by_key.items(), key=lambda item: item[1].casefold())
    ]
)
if not nodes_df.empty:
    split_labels = nodes_df["questionnaire_label"].map(split_questionnaire_label)
    nodes_df["label_acronym"] = split_labels.map(lambda item: item[0])
    nodes_df["label_full_name"] = split_labels.map(lambda item: item[1])
    nodes_df["short_label"] = nodes_df["questionnaire_label"].map(short_node_label)
    nodes_df["plot_label"] = nodes_df["questionnaire_label"].map(plot_node_label)
else:
    nodes_df["label_acronym"] = []
    nodes_df["label_full_name"] = []
    nodes_df["short_label"] = []
    nodes_df["plot_label"] = []

df["has_focal_questionnaire"] = df["focal_key"] != ""
df["has_related_questionnaire"] = df["related_key"] != ""
df["is_significant"] = df["significance_clean"] == "yes"
df["source_type_allowed"] = df["source_type_clean"].isin(ALLOWED_SOURCE_TYPES)
df["relation_value_parseable"] = df["relation_value_numeric"].notna()
df["focal_in_focal_set"] = df["focal_key"].isin(focal_keys)
df["related_in_focal_set"] = df["related_key"].isin(focal_keys)
df["related_in_manual_inclusion_set"] = df["related_key"].isin(manual_related_only_keys)
df["related_in_analysis_node_set"] = df["related_key"].isin(analysis_keys)

include_mask = (
    df["has_focal_questionnaire"]
    & df["has_related_questionnaire"]
    & df["is_significant"]
    & df["source_type_allowed"]
    & df["relation_value_parseable"]
    & df["focal_in_focal_set"]
    & df["related_in_analysis_node_set"]
)


def exclusion_reasons(row):
    reasons = []
    if not row["has_focal_questionnaire"]:
        reasons.append("missing_focal_questionnaire")
    if not row["has_related_questionnaire"]:
        reasons.append("missing_related_questionnaire")
    if not row["is_significant"]:
        reasons.append("not_significant")
    if not row["source_type_allowed"]:
        reasons.append("source_type_not_allowed")
    if not row["relation_value_parseable"]:
        reasons.append("relation_value_unparseable")
    if not row["focal_in_focal_set"]:
        reasons.append("focal_not_in_focal_set")
    if row["has_related_questionnaire"] and not row["related_in_analysis_node_set"]:
        reasons.append("related_not_in_analysis_node_set")
    return "; ".join(reasons)


eligible_edges = df.loc[include_mask].copy()
excluded_rows = df.loc[~include_mask].copy()
excluded_rows["exclusion_reason"] = excluded_rows.apply(exclusion_reasons, axis=1)

excluded_non_focal_related = df.loc[
    df["has_related_questionnaire"] & ~df["related_in_analysis_node_set"]
].copy()
excluded_non_focal_related["exclusion_reason"] = "related_not_in_analysis_node_set"

eligible_edges["focal_label"] = eligible_edges["focal_key"].map(analysis_display_by_key)
eligible_edges["related_label"] = eligible_edges["related_key"].map(analysis_display_by_key)
eligible_edges["related_node_origin"] = eligible_edges["related_key"].map(
    lambda key: "focal_questionnaire" if key in focal_keys else "manual_related_inclusion"
)
eligible_edges["edge_label"] = eligible_edges.apply(
    lambda row: format_edge_label(row["relation_value_numeric"], row["source_type_clean"]), axis=1
)
eligible_edges["edge_color"] = eligible_edges["relation_value_numeric"].map(edge_color)
eligible_edges["edge_style"] = eligible_edges["source_type_clean"].map(edge_linestyle)
eligible_edges["edge_abs_relation"] = eligible_edges["relation_value_numeric"].abs()

node_u = []
node_v = []
for _, row in eligible_edges.iterrows():
    first, second = sorted([row["focal_key"], row["related_key"]])
    node_u.append(first)
    node_v.append(second)
eligible_edges["node_u_key"] = node_u
eligible_edges["node_v_key"] = node_v
eligible_edges["node_u_label"] = eligible_edges["node_u_key"].map(analysis_display_by_key)
eligible_edges["node_v_label"] = eligible_edges["node_v_key"].map(analysis_display_by_key)

# Build graph-level edges. If repeated rows exist for the same pair, the visual edge uses the mean value,
# while cleaned_network_edges.csv preserves the row-level eligible edge list.
graph_edge_records = []
if not eligible_edges.empty:
    for (source_key, target_key), group in eligible_edges.groupby(["node_u_key", "node_v_key"], sort=False):
        mean_value = group["relation_value_numeric"].mean()
        source_types = sorted(group["source_type_clean"].dropna().unique())
        if len(group) <= 3:
            label = "; ".join(group["edge_label"].tolist())
        else:
            label = f"{format_edge_label(mean_value, group.iloc[0]['source_type_clean'])} mean; n={len(group)}"
        graph_edge_records.append(
            {
                "source_key": source_key,
                "target_key": target_key,
                "source_label": analysis_display_by_key[source_key],
                "target_label": analysis_display_by_key[target_key],
                "relation_value": mean_value,
                "abs_relation_value": abs(mean_value),
                "edge_label": label,
                "edge_color": edge_color(mean_value),
                "edge_style": edge_linestyle(group.iloc[0]["source_type_clean"]),
                "n_relations": len(group),
                "relation_source_type": "; ".join(source_types),
            }
        )

graph_edges = pd.DataFrame(graph_edge_records)

connected_keys = set()
if not graph_edges.empty:
    connected_keys.update(graph_edges["source_key"])
    connected_keys.update(graph_edges["target_key"])
nodes_df["is_connected"] = nodes_df["questionnaire_key"].isin(connected_keys)
nodes_df["degree"] = nodes_df["questionnaire_key"].map(
    lambda key: int((eligible_edges["node_u_key"].eq(key) | eligible_edges["node_v_key"].eq(key)).sum())
    if not eligible_edges.empty else 0
)

print("Number of total rows loaded:", len(df))
print("Number of unique focal questionnaires:", len(focal_keys))
print("Number of manually included related-only questionnaires:", len(manual_related_only_keys))
print("Total analysis nodes:", len(analysis_keys))
print("Number of rows with significant values:", int(df["is_significant"].sum()))
print("Number of rows with allowed relation_source_type:", int(df["source_type_allowed"].sum()))
print("Number of rows where related_questionnaire is also in the focal questionnaire set:", int(df["related_in_focal_set"].sum()))
print("Number of rows where related_questionnaire is in the analysis node set:", int(df["related_in_analysis_node_set"].sum()))
print("Final number of eligible edge rows included:", len(eligible_edges))

reason_counter = Counter()
for reasons in excluded_rows["exclusion_reason"]:
    for reason in str(reasons).split("; "):
        if reason:
            reason_counter[reason] += 1
exclusion_counts_df = pd.DataFrame(
    sorted(reason_counter.items()), columns=["exclusion_reason", "row_count"]
)
show_table(exclusion_counts_df, "\nNumber of excluded rows by exclusion reason:")

summary_rows = [
    {"metric": "total analysis nodes", "value": len(nodes_df)},
    {"metric": "focal nodes", "value": len(focal_keys)},
    {"metric": "manual related-only nodes", "value": len(manual_related_only_keys)},
    {"metric": "connected analysis nodes", "value": int(nodes_df["is_connected"].sum())},
    {"metric": "isolated analysis nodes", "value": int((~nodes_df["is_connected"]).sum())},
    {"metric": "positive eligible edge rows", "value": int((eligible_edges["relation_value_numeric"] > 0).sum())},
    {"metric": "negative eligible edge rows", "value": int((eligible_edges["relation_value_numeric"] < 0).sum())},
]
source_type_counts = (
    eligible_edges["source_type_clean"]
    .value_counts()
    .rename_axis("relation_source_type")
    .reset_index(name="edge_count")
)
summary_table = pd.concat(
    [
        pd.DataFrame(summary_rows),
        source_type_counts.rename(columns={"relation_source_type": "metric", "edge_count": "value"}),
    ],
    ignore_index=True,
)
show_table(summary_table, "\nSummary table:")


Number of total rows loaded: 288
Number of unique focal questionnaires: 21
Number of manually included related-only questionnaires: 2
Total analysis nodes: 23
Number of rows with significant values: 200
Number of rows with allowed relation_source_type: 255
Number of rows where related_questionnaire is also in the focal questionnaire set: 28
Number of rows where related_questionnaire is in the analysis node set: 38
Final number of eligible edge rows included: 32

Number of excluded rows by exclusion reason:


,exclusion_reason,row_count
0,not_significant,88
1,related_not_in_analysis_node_set,250
2,source_type_not_allowed,33



Summary table:


,metric,value
0,total analysis nodes,23
1,focal nodes,21
2,manual related-only nodes,2
3,connected analysis nodes,22
4,isolated analysis nodes,1
5,positive eligible edge rows,27
6,negative eligible edge rows,5
7,correlation_only,23
8,convergent_validity,9


## 4. Save Cleaned Tables

The edge table keeps eligible row-level relationships. The node table includes both focal nodes and the explicitly included related-only nodes. A separate node label key supports figure checking and manuscript captions.


In [4]:
preferred_edge_columns = [
    "focal_questionnaire",
    "related_questionnaire",
    "relation_value",
    "relation_value_numeric",
    "significance",
    "relation_source_type",
    "source_type_clean",
    "edge_label",
    "edge_style",
    "focal_label",
    "related_label",
    "related_node_origin",
    "node_u_label",
    "node_v_label",
    "edge_abs_relation",
]
optional_edge_columns = [column for column in ["article_reference", "notes"] if column in eligible_edges.columns]
edge_columns = [column for column in preferred_edge_columns + optional_edge_columns if column in eligible_edges.columns]

node_label_key = nodes_df[[
    "short_label",
    "plot_label",
    "questionnaire_label",
    "label_acronym",
    "label_full_name",
    "node_origin",
    "degree",
    "is_connected",
]].sort_values("short_label")

eligible_edges[edge_columns].to_csv(CLEANED_EDGES_PATH, index=False)
nodes_df.to_csv(NODE_LIST_PATH, index=False)
node_label_key.to_csv(NODE_LABEL_KEY_PATH, index=False)
excluded_rows.to_csv(EXCLUDED_ROWS_QA_PATH, index=False)
excluded_non_focal_related.to_csv(EXCLUDED_NON_FOCAL_PATH, index=False)

print(f"Saved cleaned edge list: {CLEANED_EDGES_PATH}")
print(f"Saved node list: {NODE_LIST_PATH}")
print(f"Saved node label key: {NODE_LABEL_KEY_PATH}")
print(f"Saved excluded rows QA table: {EXCLUDED_ROWS_QA_PATH}")
print(f"Saved non-analysis related questionnaire QA table: {EXCLUDED_NON_FOCAL_PATH}")


Saved cleaned edge list: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/tables/cleaned_network_edges.csv
Saved node list: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/tables/network_nodes.csv
Saved node label key: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/tables/network_node_label_key.csv
Saved excluded rows QA table: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/tables/excluded_rows_QA.csv
Saved non-analysis related questionnaire QA table: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/tables/excluded_non_focal_related_questionnai

## 5. Build the NetworkX Graph

All analysis nodes are added, including isolated nodes. Eligible significant relationships become undirected edges.


In [5]:
G = nx.Graph()
for _, row in nodes_df.iterrows():
    G.add_node(
        row["questionnaire_key"],
        label=row["questionnaire_label"],
        short_label=row["short_label"],
        plot_label=row["plot_label"],
        acronym=row["label_acronym"],
        full_name=row["label_full_name"],
        node_origin=row["node_origin"],
        degree=int(row["degree"]),
        is_connected=bool(row["is_connected"]),
    )

for _, row in graph_edges.iterrows():
    G.add_edge(
        row["source_key"],
        row["target_key"],
        relation_value=float(row["relation_value"]),
        abs_relation_value=float(row["abs_relation_value"]),
        label=row["edge_label"],
        color=row["edge_color"],
        linestyle=row["edge_style"],
        n_relations=int(row["n_relations"]),
        relation_source_type=row["relation_source_type"],
    )

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")


Graph nodes: 23
Graph edges: 30


## 6. Publication-Ready Static Network Figures

The combined figure shows the two main network clusters plus the isolated focal node. Two additional files save the self-concept cluster and essential cluster separately for clearer inspection. Node labels preserve the questionnaire acronym and full questionnaire name from the workbook.


In [6]:
# Publication-oriented matplotlib settings.
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.labelsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

POSITIVE_COLOR = "#D55E00"
NEGATIVE_COLOR = "#0072B2"
NODE_FACE = "#F8F8F8"
NODE_EDGE = "#333333"
ISOLATED_NODE_EDGE = "#A8A8A8"

components = sorted(nx.connected_components(G), key=lambda comp: (len(comp), sum(G.degree(node) for node in comp)), reverse=True)
isolated_components = [component for component in components if len(component) == 1]


def component_name(component):
    labels = " ".join(G.nodes[node]["label"].casefold() for node in component)
    if any(token in labels for token in ["essential", "iptheory", "implicit person", "true selves"]):
        return "Essentialism cluster"
    if len(component) > 1:
        return "Self-concept cluster"
    return "Isolated focal node"

cluster_components = {component_name(component): component for component in components}
self_concept_component = cluster_components.get("Self-concept cluster")
essentialism_component = cluster_components.get("Essentialism cluster")
isolated_component_nodes = set().union(*isolated_components) if isolated_components else set()

SELF_CONCEPT_LAYOUT = {
    "SCC": (0.00, 0.00),
    "SCD-R": (-1.25, -0.40),
    "Phi": (-2.30, -0.38),
    "SCD-VAR": (-0.52, -0.95),
    "SCD-SD": (-1.18, -1.58),
    "ISK": (-1.30, -2.38),
    "PSC-N": (0.78, -0.48),
    "PSC-S": (0.84, -1.28),
    "PSCQ": (1.64, -0.88),
    "MTSCS-PSC": (2.35, -0.28),
    "FSC": (2.72, -0.98),
    "MTSCS-FSC": (3.62, -0.52),
    "FSCQ": (3.76, -1.36),
}

ESSENTIALISM_LAYOUT = {
    "SES": (-1.70, 0.05),
    "ES": (-0.50, 0.05),
    "BTS": (-0.42, -0.88),
    "IPT-M": (0.44, -1.36),
    "SEB": (0.62, -0.28),
    "IPT": (1.12, 0.18),
    "IPT-I": (1.92, 0.96),
    "IPT-P": (2.82, 0.58),
    "IPT-S": (2.52, -0.36),
}

EDGE_LABEL_DISTANCE = 0.055

EDGE_LABEL_RULES = {
    frozenset(["SCD-SD", "SCD-R"]): (0.58, -0.075),
    frozenset(["SCD-SD", "SCD-VAR"]): (0.50, 0.060),
    frozenset(["SCD-R", "SCD-VAR"]): (0.50, -0.065),
    frozenset(["SCC", "SCD-VAR"]): (0.52, -EDGE_LABEL_DISTANCE),
    frozenset(["SCC", "PSC-N"]): (0.50, EDGE_LABEL_DISTANCE),
    frozenset(["SCC", "PSC-S"]): (0.48, -EDGE_LABEL_DISTANCE),
    frozenset(["PSC-N", "PSCQ"]): (0.50, EDGE_LABEL_DISTANCE),
    frozenset(["PSC-S", "PSCQ"]): (0.50, -EDGE_LABEL_DISTANCE),
    frozenset(["PSCQ", "MTSCS-PSC"]): (0.54, EDGE_LABEL_DISTANCE),
    frozenset(["FSC", "MTSCS-PSC"]): (0.48, -EDGE_LABEL_DISTANCE),
    frozenset(["MTSCS-FSC", "FSCQ"]): (0.50, EDGE_LABEL_DISTANCE),
    frozenset(["FSC", "FSCQ"]): (0.52, -EDGE_LABEL_DISTANCE),
    frozenset(["SES", "ES"]): (0.50, EDGE_LABEL_DISTANCE),
    frozenset(["BTS", "IPT-M"]): (0.50, EDGE_LABEL_DISTANCE),
    frozenset(["IPT-I", "IPT-P"]): (0.48, EDGE_LABEL_DISTANCE),
    frozenset(["IPT-P", "IPT-S"]): (0.54, EDGE_LABEL_DISTANCE),
    frozenset(["IPT-I", "IPT-S"]): (0.55, -EDGE_LABEL_DISTANCE),
    frozenset(["IPT", "IPT-P"]): (0.54, EDGE_LABEL_DISTANCE),
    frozenset(["IPT", "IPT-S"]): (0.54, EDGE_LABEL_DISTANCE),
    frozenset(["SEB", "IPT"]): (0.46, -EDGE_LABEL_DISTANCE),
}


def normalize_manual_positions(layout, subgraph, x_scale=1.0, y_scale=1.0):
    raw_pos = {}
    missing = []
    for node in subgraph.nodes:
        short = subgraph.nodes[node]["short_label"]
        if short in layout:
            raw_pos[node] = layout[short]
        else:
            missing.append(node)
    if missing:
        fallback = nx.spring_layout(subgraph.subgraph(missing), seed=42, k=1.2, iterations=1000, scale=1.0, weight=None)
        raw_pos.update(fallback)
    xs = [xy[0] for xy in raw_pos.values()]
    ys = [xy[1] for xy in raw_pos.values()]
    center_x = (min(xs) + max(xs)) / 2
    center_y = (min(ys) + max(ys)) / 2
    span_x = max(max(xs) - min(xs), 1e-6)
    span_y = max(max(ys) - min(ys), 1e-6)
    return {
        node: (((x - center_x) / span_x * 2.0 * x_scale), ((y - center_y) / span_y * 2.0 * y_scale))
        for node, (x, y) in raw_pos.items()
    }


def panel_positions(subgraph, cluster_title, x_scale=1.0, y_scale=1.0):
    if subgraph.number_of_nodes() == 1:
        return {next(iter(subgraph.nodes)): (0.0, 0.0)}
    if cluster_title == "Self-concept cluster":
        return normalize_manual_positions(SELF_CONCEPT_LAYOUT, subgraph, x_scale=x_scale, y_scale=y_scale)
    if cluster_title == "Essentialism cluster":
        return normalize_manual_positions(ESSENTIALISM_LAYOUT, subgraph, x_scale=x_scale, y_scale=y_scale)
    return nx.spring_layout(subgraph, seed=42, k=1.25, iterations=1200, scale=1.0, weight=None)


def draw_network_panel(
    ax,
    subgraph,
    title,
    label_width=28,
    node_font_size=6.2,
    edge_font_size=5.8,
    x_scale=1.0,
    y_scale=1.0,
):
    pos = panel_positions(subgraph, title, x_scale=x_scale, y_scale=y_scale)
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.axis("off")
    ax.set_xlim(-1.33 * x_scale, 1.33 * x_scale)
    ax.set_ylim(-1.33 * y_scale, 1.33 * y_scale)

    for source_type, linestyle in [
        ("correlation_only", "-"),
        ("convergent_validity", "--"),
        ("regression_semipartial correlation", ":"),
    ]:
        edgelist = [
            (u, v) for u, v, data in subgraph.edges(data=True)
            if source_type in data["relation_source_type"].split("; ")
        ]
        if not edgelist:
            continue
        widths = [0.8 + 4.8 * subgraph.edges[edge]["abs_relation_value"] for edge in edgelist]
        colors = [subgraph.edges[edge]["color"] for edge in edgelist]
        nx.draw_networkx_edges(
            subgraph,
            pos,
            edgelist=edgelist,
            edge_color=colors,
            width=widths,
            style=linestyle,
            alpha=0.80,
            ax=ax,
        )

    for i, (source, target, data) in enumerate(subgraph.edges(data=True)):
        x1, y1 = pos[source]
        x2, y2 = pos[target]
        source_short = subgraph.nodes[source]["short_label"]
        target_short = subgraph.nodes[target]["short_label"]
        factor, offset = EDGE_LABEL_RULES.get(frozenset([source_short, target_short]), (0.52, EDGE_LABEL_DISTANCE))
        lx = x1 + (x2 - x1) * factor
        ly = y1 + (y2 - y1) * factor
        dx = x2 - x1
        dy = y2 - y1
        length = math.hypot(dx, dy) or 1.0
        lx -= dy / length * offset
        ly += dx / length * offset
        ax.text(
            lx,
            ly,
            data["label"],
            color=data["color"],
            fontsize=edge_font_size,
            fontweight="bold",
            ha="center",
            va="center",
            bbox={"boxstyle": "round,pad=0.10", "facecolor": "white", "edgecolor": "none", "alpha": 0.90},
            zorder=12,
        )

    for node, data in subgraph.nodes(data=True):
        x, y = pos[node]
        node_label = plot_node_label(data["label"], width=label_width)
        ax.text(
            x,
            y,
            node_label,
            ha="center",
            va="center",
            fontsize=node_font_size,
            fontweight="bold",
            linespacing=1.12,
            bbox={
                "boxstyle": "round,pad=0.34",
                "facecolor": NODE_FACE,
                "edgecolor": NODE_EDGE if subgraph.degree(node) > 0 else ISOLATED_NODE_EDGE,
                "linewidth": 0.9,
            },
            zorder=8,
        )


def add_figure_legend(fig, y=0.035, ncol=5):
    handles = [
        Line2D([0], [0], color=POSITIVE_COLOR, lw=2.4, label="Positive relation"),
        Line2D([0], [0], color=NEGATIVE_COLOR, lw=2.4, label="Negative relation"),
        Line2D([0], [0], color="#555555", lw=1.9, linestyle="-", label="Correlation only"),
        Line2D([0], [0], color="#555555", lw=1.9, linestyle="--", label="Convergent validity"),
        Line2D([0], [0], color="#555555", lw=1.9, linestyle=":", label="Semipartial regression (sp)"),
    ]
    fig.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(0.02, y),
        frameon=False,
        fontsize=8,
        handlelength=2.5,
        ncol=ncol,
        columnspacing=1.3,
    )


def save_figure(fig, paths):
    fig.savefig(paths["png"], dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(paths["pdf"], bbox_inches="tight", facecolor="white")
    fig.savefig(paths["svg"], bbox_inches="tight", facecolor="white")
    for fmt, path in paths.items():
        print(f"Saved {fmt.upper()} figure: {path}")

# Combined figure: two large cluster panels plus isolated node panel.
fig = plt.figure(figsize=(20.5, 11.6))
fig.patch.set_facecolor("white")
grid = fig.add_gridspec(
    nrows=1,
    ncols=3,
    width_ratios=[1.42, 1.08, 0.38],
    wspace=0.14,
)
axes = [fig.add_subplot(grid[0, col]) for col in range(3)]
fig.suptitle(GRAPH_TITLE, x=0.02, y=0.985, ha="left", va="top", fontweight="bold", fontsize=14)

if self_concept_component:
    draw_network_panel(
        axes[0],
        G.subgraph(self_concept_component).copy(),
        "Self-concept cluster",
        label_width=23,
        node_font_size=5.25,
        edge_font_size=5.45,
        x_scale=1.30,
        y_scale=1.05,
    )
else:
    axes[0].axis("off")

if essentialism_component:
    draw_network_panel(
        axes[1],
        G.subgraph(essentialism_component).copy(),
        "Essentialism cluster",
        label_width=23,
        node_font_size=5.25,
        edge_font_size=5.45,
        x_scale=1.12,
        y_scale=1.00,
    )
else:
    axes[1].axis("off")

if isolated_component_nodes:
    draw_network_panel(
        axes[2],
        G.subgraph(isolated_component_nodes).copy(),
        "Isolated focal node",
        label_width=20,
        node_font_size=5.5,
        edge_font_size=5.2,
    )
else:
    axes[2].axis("off")

add_figure_legend(fig, y=0.025, ncol=5)
fig.text(
    0.02,
    0.010,
    "Edges include significant eligible relation values only. Width scales with |r|; node labels preserve the questionnaire acronym and full name from the workbook.",
    ha="left",
    va="bottom",
    fontsize=8,
    color="#333333",
)
fig.subplots_adjust(left=0.02, right=0.985, top=0.91, bottom=0.10)
save_figure(fig, COMBINED_GRAPH_PATHS)
plt.show()

# Separate self-concept cluster figure.
if self_concept_component:
    fig_self, ax_self = plt.subplots(figsize=(17.2, 12.4))
    fig_self.patch.set_facecolor("white")
    draw_network_panel(
        ax_self,
        G.subgraph(self_concept_component).copy(),
        "Self-concept cluster",
        label_width=32,
        node_font_size=7.0,
        edge_font_size=6.9,
        x_scale=1.34,
        y_scale=1.08,
    )
    add_figure_legend(fig_self, y=0.025, ncol=5)
    fig_self.subplots_adjust(left=0.03, right=0.98, top=0.93, bottom=0.10)
    save_figure(fig_self, SELF_CONCEPT_CLUSTER_PATHS)
    plt.show()

# Separate essentialism cluster figure.
if essentialism_component:
    fig_essential, ax_essential = plt.subplots(figsize=(14.0, 10.4))
    fig_essential.patch.set_facecolor("white")
    draw_network_panel(
        ax_essential,
        G.subgraph(essentialism_component).copy(),
        "Essentialism cluster",
        label_width=32,
        node_font_size=7.0,
        edge_font_size=6.9,
        x_scale=1.12,
        y_scale=1.00,
    )
    add_figure_legend(fig_essential, y=0.025, ncol=5)
    fig_essential.subplots_adjust(left=0.03, right=0.98, top=0.93, bottom=0.10)
    save_figure(fig_essential, ESSENTIALISM_CLUSTER_PATHS)
    plt.show()


Saved PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.png
Saved PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.pdf
Saved SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.svg


/var/folders/_c/vk6v15cs2d79cy5rk79bkrrm0000gn/T/ipykernel_13194/3650852217.py:303: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self_concept_cluster_network.png
Saved PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self_concept_cluster_network.pdf
Saved SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self_concept_cluster_network.svg


/var/folders/_c/vk6v15cs2d79cy5rk79bkrrm0000gn/T/ipykernel_13194/3650852217.py:322: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/essentialism_cluster_network.png
Saved PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/essentialism_cluster_network.pdf
Saved SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/essentialism_cluster_network.svg


/var/folders/_c/vk6v15cs2d79cy5rk79bkrrm0000gn/T/ipykernel_13194/3650852217.py:341: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Output Review

The files below are the expected reproducible outputs from the notebook. No interactive HTML graph is generated.


In [7]:
expected_paths = [
    *COMBINED_GRAPH_PATHS.values(),
    *SELF_CONCEPT_CLUSTER_PATHS.values(),
    *ESSENTIALISM_CLUSTER_PATHS.values(),
    CLEANED_EDGES_PATH,
    NODE_LIST_PATH,
    NODE_LABEL_KEY_PATH,
    EXCLUDED_ROWS_QA_PATH,
    EXCLUDED_NON_FOCAL_PATH,
]
for path in expected_paths:
    print(path, "exists=", path.exists())


/Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.png exists= True
/Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.pdf exists= True
/Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self questionnaires mapping network.svg exists= True
/Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self_concept_cluster_network.png exists= True
/Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/self_concept_cluster_network.pdf exists= True
/Users/dotanshani/D

## 8. Tier 1+2 Louvain Network Analysis

This section builds the expanded Tier 1+2 questionnaire network from `database_validity_extraction.xlsx`. It preserves the Tier 1 visual conventions, excludes discriminant/divergent validity rows, includes the requested related-only nodes, collapses duplicate pair edges by keeping the largest absolute correlation, and assigns clusters using weighted Louvain community detection.


In [8]:
# Tier 1+2 network configuration.
plt.switch_backend("Agg")
import shutil

TIER12_INPUT_FILENAME = "database_validity_extraction.xlsx"
TIER12_DATA_PATH = PROJECT_ROOT / "data" / TIER12_INPUT_FILENAME
TIER12_GRAPH_TITLE = "Tier 1 and 2 self questionnaires mapping network"

TIER12_COMBINED_GRAPH_PATHS = {
    "png": OUTPUT_FIGURES_DIR / "tier_1_2_questionnaires_mapping_network.png",
    "pdf": OUTPUT_FIGURES_DIR / "tier_1_2_questionnaires_mapping_network.pdf",
    "svg": OUTPUT_FIGURES_DIR / "tier_1_2_questionnaires_mapping_network.svg",
}
TIER12_MAIN_NETWORK_GRAPH_PATHS = {
    "png": OUTPUT_FIGURES_DIR / "tier_1_2_main_network_questionnaires_mapping_network.png",
    "pdf": OUTPUT_FIGURES_DIR / "tier_1_2_main_network_questionnaires_mapping_network.pdf",
    "svg": OUTPUT_FIGURES_DIR / "tier_1_2_main_network_questionnaires_mapping_network.svg",
}

TIER12_CLEANED_EDGES_PATH = OUTPUT_TABLES_DIR / "tier_1_2_cleaned_network_edges.csv"
TIER12_DUPLICATE_EDGES_QA_PATH = OUTPUT_TABLES_DIR / "tier_1_2_duplicate_pair_edges_QA.csv"
TIER12_NODE_LIST_PATH = OUTPUT_TABLES_DIR / "tier_1_2_network_nodes.csv"
TIER12_CLUSTER_MEMBERSHIP_PATH = OUTPUT_TABLES_DIR / "tier_1_2_cluster_membership.csv"
TIER12_ISOLATED_NODES_PATH = OUTPUT_TABLES_DIR / "tier_1_2_isolated_nodes.csv"
TIER12_EXCLUDED_ROWS_QA_PATH = OUTPUT_TABLES_DIR / "tier_1_2_excluded_rows_QA.csv"
TIER12_EXCLUDED_NON_NETWORK_PATH = OUTPUT_TABLES_DIR / "tier_1_2_excluded_non_network_questionnaires.csv"
TIER12_NODE_CENTRALITY_XLSX_PATH = OUTPUT_TABLES_DIR / "tier_1_2_node_centrality.xlsx"
TIER12_TOP25_ABS_STRENGTH_XLSX_PATH = OUTPUT_TABLES_DIR / "tier_1_2_top25_absolute_weighted_node_strength_by_cluster.xlsx"
TIER12_TOP25_BRIDGE_STRENGTH_XLSX_PATH = OUTPUT_TABLES_DIR / "tier_1_2_top25_bridge_strength_by_cluster.xlsx"
TIER12_TOP25_WITHIN_STRENGTH_XLSX_PATH = OUTPUT_TABLES_DIR / "tier_1_2_top25_within_cluster_strength_by_cluster.xlsx"

TIER12_EXCLUDED_SOURCE_TYPES = {
    "discriminant_validity",
    "discriminant validity",
    "divergent_validity",
}
TIER12_REGRESSION_SOURCE_TYPES = {
    "regression_semipartial",
    "regression_semipartial correlation",
}
TIER12_SOURCE_TYPE_STYLES = {
    "correlation_only": "-",
    "convergent_validity": "--",
    "regression_semipartial": ":",
    "regression_semipartial correlation": ":",
    "criterion_validity": "-.",
}
TIER12_EDGE_LABEL_DISTANCE = EDGE_LABEL_DISTANCE
TIER12_LAYOUT_SEED = 42
TIER12_LOUVAIN_RESOLUTION = 1.0
TIER12_MIN_CLUSTER_FIGURE_NODES = 4

TIER12_ADDITIONAL_RELATED_QUESTIONNAIRES = [
    "(PSCQ): Past Self-Continuity Questionnaire",
    "(ES): Essentialist Beliefs about Human Kinds Scale",
    "(IPTheory - S): Implicit person theory measure - specific abilities",
    "(HVIC - I): Horizontal and Vertical Individualism and Collectivism Scale - Individualism scale",
    "(HVIC - C): Horizontal and Vertical Individualism and Collectivism Scale - Collectivism scale",
    "(SAS): Sheldon's five-item Authenticity Scale",
    "(SOAS): Sense of Authenticity Scale",
    "(RSKS): Reflective Self-Knowledge Scale",
    "(ESKS): Experiential Self-Knowledge Scale",
    "(SCD - CSD): Self-Concept Differentiation index - corrected cross-role standard deviation",
    "(SEQ): Self Experiences Questionnaire",
    "(EQ - R): Experiences Questionnaire - Rumination subscale",
    "(SEQ): Self Experiences Questionnaire",
    "(CSES): Contingent Self-Esteem Scale",
    "(RS): Rigidity Scale",
    "(DS): Dogmatism Scale",
    "(PFI): Personal Fear of Invalidity Scale",
    "(TOA ): Tolerance for Ambiguity Scale (Lorsch & Morse)",
    "(ES-SG): Essentialist Beliefs about Social Groups scale",
    "(TOA): Intolerance of Ambiguity Scale (Martin & Westie)",
    "(SAM - SK): Self-Awareness Questionnaire - Self-Knowledge subscale",
    "(MQ): Mindfulness Questionnaire",
]
TIER12_MANUAL_TIER_OVERRIDES = {
    questionnaire_key("(PSCQ): Past Self-Continuity Questionnaire"): 1,
    questionnaire_key("(ES): Essentialist Beliefs about Human Kinds Scale"): 1,
    questionnaire_key("(IPTheory - S): Implicit person theory measure - specific abilities"): 1,
    questionnaire_key("(SCD - CSD): Self-Concept Differentiation index - corrected cross-role standard deviation"): 1,
    questionnaire_key("(ES-SG): Essentialist Beliefs about Social Groups scale"): 1,
}
TIER12_MANUAL_NODE_KEYS = {
    questionnaire_key(label): clean_text(label)
    for label in TIER12_ADDITIONAL_RELATED_QUESTIONNAIRES
}

if not TIER12_DATA_PATH.exists():
    raise FileNotFoundError(f"Expected Tier 1+2 input file not found: {TIER12_DATA_PATH}")

tier12_xlsx = pd.ExcelFile(TIER12_DATA_PATH)
tier12_sheet_name = PREFERRED_SHEET if PREFERRED_SHEET in tier12_xlsx.sheet_names else tier12_xlsx.sheet_names[0]
tier12_raw = pd.read_excel(TIER12_DATA_PATH, sheet_name=tier12_sheet_name)

print(f"Tier 1+2 workbook: {TIER12_DATA_PATH}")
print(f"Available sheets: {tier12_xlsx.sheet_names}")
print(f"Selected sheet: {tier12_sheet_name}")
print("Actual Tier 1+2 column names:")
for i, column in enumerate(tier12_raw.columns, start=1):
    print(f"  {i}. {column!r}")

TIER12_REQUIRED_COLUMNS = [
    "focal_questionnaire",
    "Tier",
    "related_questionnaire",
    "relation_value",
    "significance",
    "relation_source_type",
]

tier12_actual_by_canonical = {}
for column in tier12_raw.columns:
    tier12_actual_by_canonical.setdefault(canonicalize_column_name(column), column)

tier12_missing_columns = [
    column for column in TIER12_REQUIRED_COLUMNS
    if canonicalize_column_name(column) not in tier12_actual_by_canonical
]
if tier12_missing_columns:
    raise ValueError(
        "Missing Tier 1+2 required columns after minor normalization: "
        + ", ".join(tier12_missing_columns)
        + "\nActual columns were: "
        + ", ".join(map(str, tier12_raw.columns))
    )

tier12_column_mapping = {
    tier12_actual_by_canonical[canonicalize_column_name(expected)]: expected
    for expected in TIER12_REQUIRED_COLUMNS
}
tier12_df = tier12_raw.rename(columns=tier12_column_mapping).copy()

print("\nTier 1+2 column mapping used:")
for actual, expected in tier12_column_mapping.items():
    print(f"  {actual!r} -> {expected!r}")


def parse_tier(value):
    text = clean_text(value)
    if not text:
        return pd.NA
    match = re.search(r"\d+", text)
    if not match:
        return pd.NA
    tier_value = int(match.group(0))
    return tier_value if tier_value in {1, 2} else pd.NA


def valid_correlation(value):
    parsed = parse_relation_value(value)
    if pd.isna(parsed):
        return pd.NA
    parsed = float(parsed)
    if abs(parsed) > 1:
        return pd.NA
    return parsed


def normalize_source_type(value):
    return re.sub(r"\s+", " ", clean_text(value).casefold())


def tier12_edge_linestyle(source_type):
    source_type = normalize_source_type(source_type)
    return TIER12_SOURCE_TYPE_STYLES.get(source_type, "-")


def tier12_format_edge_label(value, source_type):
    number = format_relation_number(value)
    if normalize_source_type(source_type) in TIER12_REGRESSION_SOURCE_TYPES:
        return f"sp={number}"
    return f"r={number}"


def tier12_slugify(value, max_length=46):
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    text = re.sub(r"_+", "_", text)
    return (text[:max_length].strip("_") or "cluster")


def source_type_display(source_type):
    source_type = normalize_source_type(source_type)
    if source_type == "correlation_only":
        return "Correlation only"
    if source_type == "convergent_validity":
        return "Convergent validity"
    if source_type in TIER12_REGRESSION_SOURCE_TYPES:
        return "Semipartial regression (sp)"
    if source_type == "criterion_validity":
        return "Criterion validity"
    return clean_text(source_type).replace("_", " ").title()

# Clean and parse fields.
tier12_df["original_row"] = tier12_df.index + 2
tier12_df["focal_clean"] = tier12_df["focal_questionnaire"].map(clean_text)
tier12_df["related_clean"] = tier12_df["related_questionnaire"].map(clean_text)
tier12_df["focal_key"] = tier12_df["focal_clean"].map(questionnaire_key)
tier12_df["related_key"] = tier12_df["related_clean"].map(questionnaire_key)
tier12_df["tier_numeric"] = tier12_df["Tier"].map(parse_tier)
tier12_df["significance_clean"] = tier12_df["significance"].map(lambda value: clean_text(value).casefold())
tier12_df["source_type_clean"] = tier12_df["relation_source_type"].map(normalize_source_type)
tier12_df["relation_value_numeric"] = pd.to_numeric(tier12_df["relation_value"].map(valid_correlation), errors="coerce")
tier12_df["relation_abs"] = tier12_df["relation_value_numeric"].abs()
tier12_df["source_type_not_excluded"] = (
    tier12_df["source_type_clean"].ne("")
    & ~tier12_df["source_type_clean"].isin(TIER12_EXCLUDED_SOURCE_TYPES)
)
tier12_df["significance_is_yes"] = tier12_df["significance_clean"].eq("yes")
tier12_df["significance_is_no"] = tier12_df["significance_clean"].eq("no")
tier12_df["significance_is_unclear"] = ~tier12_df["significance_clean"].isin(["yes", "no"])
tier12_df["significance_allowed"] = (
    tier12_df["significance_is_yes"]
    | (tier12_df["significance_is_unclear"] & tier12_df["relation_abs"].ge(0.02))
)

# Node universe: focal Tier 1/2 nodes with at least one non-excluded relationship row,
# plus manually requested related-only nodes.
tier12_focal_node_rows = tier12_df[
    tier12_df["source_type_not_excluded"]
    & tier12_df["tier_numeric"].isin([1, 2])
    & tier12_df["focal_key"].ne("")
].copy()

tier12_node_records_by_key = {}
for _, row in tier12_focal_node_rows.iterrows():
    key = row["focal_key"]
    if key not in tier12_node_records_by_key:
        tier12_node_records_by_key[key] = {
            "node_key": key,
            "questionnaire": row["focal_clean"],
            "tier": int(row["tier_numeric"]),
            "node_origin": "focal_questionnaire",
        }

related_label_by_key = (
    tier12_df[tier12_df["related_key"].ne("")]
    .drop_duplicates("related_key")
    .set_index("related_key")["related_clean"]
    .to_dict()
)
manual_related_found = []
manual_related_missing = []
for key, requested_label in TIER12_MANUAL_NODE_KEYS.items():
    label = related_label_by_key.get(key, requested_label)
    manual_tier = TIER12_MANUAL_TIER_OVERRIDES.get(key, 2)
    if key in related_label_by_key:
        manual_related_found.append(label)
    else:
        manual_related_missing.append(requested_label)
    if key in tier12_node_records_by_key:
        tier12_node_records_by_key[key]["tier"] = manual_tier
        if tier12_node_records_by_key[key]["node_origin"] != "focal_questionnaire":
            tier12_node_records_by_key[key]["node_origin"] = "manual_related_inclusion"
        else:
            tier12_node_records_by_key[key]["node_origin"] = "focal_questionnaire; manual_tier_override"
    else:
        tier12_node_records_by_key[key] = {
            "node_key": key,
            "questionnaire": label,
            "tier": manual_tier,
            "node_origin": "manual_related_inclusion",
        }

# Force manual Tier overrides even if the node was already focal in the database.
for key, manual_tier in TIER12_MANUAL_TIER_OVERRIDES.items():
    if key in tier12_node_records_by_key:
        tier12_node_records_by_key[key]["tier"] = manual_tier

network_node_keys = set(tier12_node_records_by_key)

tier12_df["focal_in_node_universe"] = tier12_df["focal_key"].isin(network_node_keys)
tier12_df["related_in_node_universe"] = tier12_df["related_key"].isin(network_node_keys)
tier12_df["numeric_relation_value"] = tier12_df["relation_value_numeric"].notna()
tier12_df["eligible_pre_pair_aggregation"] = (
    tier12_df["source_type_not_excluded"]
    & tier12_df["numeric_relation_value"]
    & tier12_df["significance_allowed"]
    & tier12_df["focal_in_node_universe"]
    & tier12_df["related_in_node_universe"]
    & tier12_df["focal_key"].ne(tier12_df["related_key"])
)

tier12_edge_candidates = tier12_df[tier12_df["eligible_pre_pair_aggregation"]].copy()
tier12_edge_candidates["node_u_key"] = tier12_edge_candidates.apply(
    lambda row: sorted([row["focal_key"], row["related_key"]])[0], axis=1
)
tier12_edge_candidates["node_v_key"] = tier12_edge_candidates.apply(
    lambda row: sorted([row["focal_key"], row["related_key"]])[1], axis=1
)
tier12_edge_candidates["pair_key"] = tier12_edge_candidates["node_u_key"] + " || " + tier12_edge_candidates["node_v_key"]

tier12_selected_edges = (
    tier12_edge_candidates
    .sort_values(["pair_key", "relation_abs", "original_row"], ascending=[True, False, True])
    .groupby("pair_key", as_index=False, sort=False)
    .head(1)
    .copy()
)
tier12_selected_edge_indices = set(tier12_selected_edges.index)
tier12_duplicate_edges_qa = tier12_edge_candidates[~tier12_edge_candidates.index.isin(tier12_selected_edge_indices)].copy()

selected_pair_details = tier12_selected_edges[[
    "pair_key",
    "relation_value_numeric",
    "relation_abs",
    "source_type_clean",
    "original_row",
]].rename(columns={
    "relation_value_numeric": "selected_relation_value_numeric",
    "relation_abs": "selected_relation_abs",
    "source_type_clean": "selected_source_type_clean",
    "original_row": "selected_original_row",
})
tier12_duplicate_edges_qa = tier12_duplicate_edges_qa.merge(selected_pair_details, on="pair_key", how="left")

tier12_nodes = pd.DataFrame(tier12_node_records_by_key.values()).sort_values(["tier", "questionnaire"]).reset_index(drop=True)
tier12_nodes["short_label"] = tier12_nodes["questionnaire"].map(short_node_label)
tier12_nodes["plot_label"] = tier12_nodes["questionnaire"].map(plot_node_label)

# Build the selected-edge graph used for clustering and visualization.
tier12_graph = nx.Graph()
for row in tier12_nodes.itertuples(index=False):
    tier12_graph.add_node(
        row.node_key,
        label=row.questionnaire,
        short_label=row.short_label,
        plot_label=row.plot_label,
        tier=int(row.tier),
        node_origin=row.node_origin,
    )

for _, row in tier12_selected_edges.iterrows():
    node_u_key, node_v_key = row["node_u_key"], row["node_v_key"]
    relation_value = float(row["relation_value_numeric"])
    source_type = row["source_type_clean"]
    tier12_graph.add_edge(
        node_u_key,
        node_v_key,
        relation_value=relation_value,
        abs_relation_value=abs(relation_value),
        weight=abs(relation_value),
        relation_source_type=source_type,
        source_type_display=source_type_display(source_type),
        edge_label=tier12_format_edge_label(relation_value, source_type),
        edge_style=tier12_edge_linestyle(source_type),
        color=edge_color(relation_value),
        focal_questionnaire=row["focal_clean"],
        related_questionnaire=row["related_clean"],
        relation_value_raw=row["relation_value"],
        significance=row["significance"],
        significance_clean=row["significance_clean"],
        original_row=int(row["original_row"]),
        pair_key=row["pair_key"],
    )

connected_tier12_nodes = [node for node in tier12_graph.nodes if tier12_graph.degree(node) > 0]
tier12_connected_graph = tier12_graph.subgraph(connected_tier12_nodes).copy()
tier12_isolated_nodes = [node for node in tier12_graph.nodes if tier12_graph.degree(node) == 0]

from networkx.algorithms import community as nx_community

if not hasattr(nx_community, "louvain_communities"):
    raise RuntimeError("networkx.algorithms.community.louvain_communities is required. Install networkx>=3.0.")

tier12_communities = list(nx_community.louvain_communities(
    tier12_connected_graph,
    weight="weight",
    seed=TIER12_LAYOUT_SEED,
    resolution=TIER12_LOUVAIN_RESOLUTION,
))
tier12_communities = sorted(
    tier12_communities,
    key=lambda nodes: (len(nodes), sum(tier12_connected_graph.degree(node) for node in nodes)),
    reverse=True,
)

tier12_cluster_by_node = {}
tier12_cluster_records = []
for cluster_id, nodes in enumerate(tier12_communities, start=1):
    sorted_nodes = sorted(
        nodes,
        key=lambda node: (-tier12_connected_graph.degree(node), tier12_graph.nodes[node]["short_label"]),
    )
    top_labels = [tier12_graph.nodes[node]["short_label"] for node in sorted_nodes[:4]]
    cluster_name = f"Cluster {cluster_id:02d}: " + ", ".join(top_labels)
    cluster_slug = tier12_slugify("_".join(top_labels[:3]))
    for node in nodes:
        tier12_cluster_by_node[node] = cluster_id
        tier12_graph.nodes[node]["cluster_id"] = cluster_id
        tier12_graph.nodes[node]["cluster_name"] = cluster_name
        tier12_graph.nodes[node]["cluster_slug"] = cluster_slug
    tier12_cluster_records.append({
        "cluster_id": cluster_id,
        "cluster_name": cluster_name,
        "cluster_slug": cluster_slug,
        "node_count": len(nodes),
        "edge_count_within_cluster": tier12_connected_graph.subgraph(nodes).number_of_edges(),
    })

for node in tier12_isolated_nodes:
    tier12_graph.nodes[node]["cluster_id"] = pd.NA
    tier12_graph.nodes[node]["cluster_name"] = "Isolated node"
    tier12_graph.nodes[node]["cluster_slug"] = "isolated"

# Rebuild the connected graph after cluster attributes are assigned so figure node rims
# can use the Louvain cluster colors.
tier12_connected_graph = tier12_graph.subgraph(connected_tier12_nodes).copy()

tier12_cluster_summary = pd.DataFrame(tier12_cluster_records)

for _, row in tier12_nodes.iterrows():
    node = row["node_key"]
    tier12_nodes.loc[tier12_nodes["node_key"] == node, "degree"] = tier12_graph.degree(node)
    tier12_nodes.loc[tier12_nodes["node_key"] == node, "cluster_id"] = tier12_graph.nodes[node].get("cluster_id", pd.NA)
    tier12_nodes.loc[tier12_nodes["node_key"] == node, "cluster_name"] = tier12_graph.nodes[node].get("cluster_name", "Isolated node")

tier12_nodes["degree"] = tier12_nodes["degree"].astype(int)
tier12_nodes["is_connected"] = tier12_nodes["degree"].gt(0)
tier12_cluster_membership = tier12_nodes[tier12_nodes["is_connected"]].copy()
tier12_isolated_nodes_table = tier12_nodes[~tier12_nodes["is_connected"]].copy()

def tier12_exclusion_reasons(row):
    reasons = []
    if not row["focal_clean"]:
        reasons.append("missing_focal_questionnaire")
    if not row["related_clean"]:
        reasons.append("missing_related_questionnaire")
    if pd.isna(row["tier_numeric"]):
        reasons.append("focal_tier_not_1_or_2_or_missing")
    if not row["source_type_not_excluded"]:
        reasons.append("excluded_or_missing_relation_source_type")
    if not row["numeric_relation_value"]:
        reasons.append("unparseable_or_out_of_range_relation_value")
    if row["significance_is_no"]:
        reasons.append("explicit_not_significant")
    if row["significance_is_unclear"] and not row["significance_allowed"]:
        reasons.append("unclear_significance_below_abs_.02")
    if row["source_type_not_excluded"] and row["numeric_relation_value"] and row["significance_allowed"]:
        if not row["focal_in_node_universe"]:
            reasons.append("focal_not_in_node_universe")
        if not row["related_in_node_universe"]:
            reasons.append("related_not_in_node_universe")
        if row["focal_key"] == row["related_key"]:
            reasons.append("self_loop")
    return "; ".join(reasons)

tier12_df["exclusion_reason"] = tier12_df.apply(tier12_exclusion_reasons, axis=1)
if len(tier12_duplicate_edges_qa):
    duplicate_reason_by_index = {idx: "duplicate_pair_lower_abs_relation" for idx in tier12_duplicate_edges_qa.index}
    tier12_df.loc[list(duplicate_reason_by_index), "exclusion_reason"] = tier12_df.loc[list(duplicate_reason_by_index), "exclusion_reason"].map(
        lambda text: "; ".join([part for part in [text, "duplicate_pair_lower_abs_relation"] if part])
    )

tier12_selected_indices = set(tier12_selected_edges.index)
tier12_excluded_rows_qa = tier12_df[~tier12_df.index.isin(tier12_selected_indices)].copy()
tier12_excluded_rows_qa = tier12_excluded_rows_qa[tier12_excluded_rows_qa["exclusion_reason"].ne("")].copy()

tier12_excluded_non_network = tier12_df[
    tier12_df["source_type_not_excluded"]
    & tier12_df["numeric_relation_value"]
    & tier12_df["significance_allowed"]
    & ~tier12_df["related_in_node_universe"]
].copy()

# Prepare clean table exports.
def node_label_from_key(node_key):
    return tier12_graph.nodes[node_key]["label"]

cleaned_tier12_edges = []
for u, v, data in tier12_graph.edges(data=True):
    cleaned_tier12_edges.append({
        "node_u_key": u,
        "node_v_key": v,
        "node_u_label": node_label_from_key(u),
        "node_v_label": node_label_from_key(v),
        "node_u_tier": tier12_graph.nodes[u]["tier"],
        "node_v_tier": tier12_graph.nodes[v]["tier"],
        "node_u_cluster_id": tier12_graph.nodes[u].get("cluster_id", pd.NA),
        "node_v_cluster_id": tier12_graph.nodes[v].get("cluster_id", pd.NA),
        "relation_value": data["relation_value"],
        "abs_relation_value": data["abs_relation_value"],
        "edge_label": data["edge_label"],
        "relation_source_type": data["relation_source_type"],
        "edge_style": data["edge_style"],
        "significance": data["significance"],
        "original_row": data["original_row"],
        "pair_key": data["pair_key"],
        "focal_questionnaire": data["focal_questionnaire"],
        "related_questionnaire": data["related_questionnaire"],
        "relation_value_raw": data["relation_value_raw"],
    })
cleaned_tier12_edges = pd.DataFrame(cleaned_tier12_edges).sort_values(
    ["node_u_cluster_id", "node_v_cluster_id", "node_u_label", "node_v_label"]
)


# Weighted node centrality metrics. Absolute weighted node strength is the main
# structural centrality index: sum(abs(r)) across a node's incident edges.
def tier12_node_centrality_records(graph):
    records = []
    for node, attrs in graph.nodes(data=True):
        node_cluster_id = attrs.get("cluster_id", pd.NA)
        degree = graph.degree(node)
        absolute_weighted_node_strength = 0.0
        expected_influence_signed = 0.0
        positive_strength_abs = 0.0
        negative_strength_abs = 0.0
        bridge_strength_abs = 0.0
        within_cluster_strength_abs = 0.0
        bridge_degree = 0
        within_cluster_degree = 0
        positive_edge_count = 0
        negative_edge_count = 0

        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)
            relation_value = float(edge_data["relation_value"])
            abs_relation_value = abs(relation_value)
            neighbor_cluster_id = graph.nodes[neighbor].get("cluster_id", pd.NA)

            absolute_weighted_node_strength += abs_relation_value
            expected_influence_signed += relation_value
            if relation_value > 0:
                positive_strength_abs += abs_relation_value
                positive_edge_count += 1
            elif relation_value < 0:
                negative_strength_abs += abs_relation_value
                negative_edge_count += 1

            same_cluster = (
                not pd.isna(node_cluster_id)
                and not pd.isna(neighbor_cluster_id)
                and int(node_cluster_id) == int(neighbor_cluster_id)
            )
            if same_cluster:
                within_cluster_strength_abs += abs_relation_value
                within_cluster_degree += 1
            elif not pd.isna(node_cluster_id) and not pd.isna(neighbor_cluster_id):
                bridge_strength_abs += abs_relation_value
                bridge_degree += 1

        records.append({
            "node_key": node,
            "questionnaire": attrs["label"],
            "short_label": attrs["short_label"],
            "tier": attrs["tier"],
            "node_origin": attrs["node_origin"],
            "cluster_id": node_cluster_id,
            "cluster_name": attrs.get("cluster_name", "Isolated node"),
            "cluster_slug": attrs.get("cluster_slug", "isolated"),
            "is_connected": degree > 0,
            "degree": degree,
            "positive_edge_count": positive_edge_count,
            "negative_edge_count": negative_edge_count,
            "absolute_weighted_node_strength": absolute_weighted_node_strength,
            "mean_abs_relation": absolute_weighted_node_strength / degree if degree else 0.0,
            "expected_influence_signed": expected_influence_signed,
            "positive_strength_abs": positive_strength_abs,
            "negative_strength_abs": negative_strength_abs,
            "bridge_degree": bridge_degree,
            "bridge_strength_abs": bridge_strength_abs,
            "within_cluster_degree": within_cluster_degree,
            "within_cluster_strength_abs": within_cluster_strength_abs,
            "bridge_strength_share": (
                bridge_strength_abs / absolute_weighted_node_strength
                if absolute_weighted_node_strength else 0.0
            ),
        })
    return records


def add_cluster_rank(table, metric, rank_column):
    table[rank_column] = (
        table
        .groupby("cluster_id", dropna=True)[metric]
        .rank(method="min", ascending=False)
        .astype("Int64")
    )
    return table


def top_25_percent_by_cluster(table, metric):
    selected_groups = []
    clustered = table[table["is_connected"] & table["cluster_id"].notna()].copy()
    for cluster_id, cluster_table in clustered.groupby("cluster_id", sort=True):
        cluster_table = cluster_table.sort_values(
            [metric, "absolute_weighted_node_strength", "degree", "short_label"],
            ascending=[False, False, False, True],
        ).reset_index(drop=True)
        top_n = max(1, math.ceil(len(cluster_table) * 0.25))
        selected = cluster_table.head(top_n).copy()
        selected.insert(0, "metric_used_for_selection", metric)
        selected.insert(1, "selected_metric_value", selected[metric])
        selected.insert(2, "rank_within_cluster_for_metric", range(1, len(selected) + 1))
        selected.insert(3, "top_25_percent_n_for_cluster", top_n)
        selected.insert(4, "cluster_node_count", len(cluster_table))
        selected_groups.append(selected)
    if not selected_groups:
        return pd.DataFrame(columns=[
            "metric_used_for_selection",
            "selected_metric_value",
            "rank_within_cluster_for_metric",
            "top_25_percent_n_for_cluster",
            "cluster_node_count",
            *table.columns,
        ])
    return pd.concat(selected_groups, ignore_index=True)


def save_excel_table(table, path, sheet_name):
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=sheet_name)


tier12_node_centrality = pd.DataFrame(tier12_node_centrality_records(tier12_graph))
tier12_node_centrality["overall_absolute_strength_rank"] = (
    tier12_node_centrality["absolute_weighted_node_strength"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
tier12_node_centrality = add_cluster_rank(
    tier12_node_centrality,
    "absolute_weighted_node_strength",
    "cluster_absolute_strength_rank",
)
tier12_node_centrality = add_cluster_rank(
    tier12_node_centrality,
    "bridge_strength_abs",
    "cluster_bridge_strength_rank",
)
tier12_node_centrality = add_cluster_rank(
    tier12_node_centrality,
    "within_cluster_strength_abs",
    "cluster_within_strength_rank",
)
tier12_node_centrality = tier12_node_centrality.sort_values(
    ["absolute_weighted_node_strength", "degree", "short_label"],
    ascending=[False, False, True],
).reset_index(drop=True)

tier12_top25_abs_strength_by_cluster = top_25_percent_by_cluster(
    tier12_node_centrality,
    "absolute_weighted_node_strength",
)
tier12_top25_bridge_strength_by_cluster = top_25_percent_by_cluster(
    tier12_node_centrality,
    "bridge_strength_abs",
)
tier12_top25_within_strength_by_cluster = top_25_percent_by_cluster(
    tier12_node_centrality,
    "within_cluster_strength_abs",
)

tier12_duplicate_edges_qa = tier12_duplicate_edges_qa.sort_values(["pair_key", "relation_abs", "original_row"], ascending=[True, False, True])

tier12_export_columns = [
    "original_row",
    "focal_questionnaire",
    "Tier",
    "related_questionnaire",
    "relation_value",
    "relation_value_numeric",
    "relation_abs",
    "significance",
    "significance_clean",
    "relation_source_type",
    "source_type_clean",
    "focal_in_node_universe",
    "related_in_node_universe",
    "exclusion_reason",
    "article_reference",
    "notes",
]
tier12_export_columns = [column for column in tier12_export_columns if column in tier12_df.columns]

tier12_nodes.to_csv(TIER12_NODE_LIST_PATH, index=False)
tier12_cluster_membership.to_csv(TIER12_CLUSTER_MEMBERSHIP_PATH, index=False)
tier12_isolated_nodes_table.to_csv(TIER12_ISOLATED_NODES_PATH, index=False)
cleaned_tier12_edges.to_csv(TIER12_CLEANED_EDGES_PATH, index=False)
tier12_duplicate_edges_qa.to_csv(TIER12_DUPLICATE_EDGES_QA_PATH, index=False)
tier12_excluded_rows_qa[tier12_export_columns].to_csv(TIER12_EXCLUDED_ROWS_QA_PATH, index=False)
tier12_excluded_non_network[tier12_export_columns].to_csv(TIER12_EXCLUDED_NON_NETWORK_PATH, index=False)
with pd.ExcelWriter(TIER12_NODE_CENTRALITY_XLSX_PATH, engine="openpyxl") as writer:
    tier12_node_centrality.to_excel(writer, index=False, sheet_name="all_nodes")
    tier12_top25_abs_strength_by_cluster.to_excel(writer, index=False, sheet_name="top25_abs_strength")
    tier12_top25_bridge_strength_by_cluster.to_excel(writer, index=False, sheet_name="top25_bridge_strength")
    tier12_top25_within_strength_by_cluster.to_excel(writer, index=False, sheet_name="top25_within_strength")
save_excel_table(
    tier12_top25_abs_strength_by_cluster,
    TIER12_TOP25_ABS_STRENGTH_XLSX_PATH,
    "top25_abs_strength",
)
save_excel_table(
    tier12_top25_bridge_strength_by_cluster,
    TIER12_TOP25_BRIDGE_STRENGTH_XLSX_PATH,
    "top25_bridge_strength",
)
save_excel_table(
    tier12_top25_within_strength_by_cluster,
    TIER12_TOP25_WITHIN_STRENGTH_XLSX_PATH,
    "top25_within_strength",
)

print("\nTier 1+2 QA checks")
print("Total rows loaded:", len(tier12_df))
print("Tier 1 rows:", int(tier12_df["tier_numeric"].eq(1).sum()))
print("Tier 2 rows:", int(tier12_df["tier_numeric"].eq(2).sum()))
print("Network nodes total:", tier12_graph.number_of_nodes())
print("Connected nodes:", len(connected_tier12_nodes))
print("Isolated nodes:", len(tier12_isolated_nodes))
print("Eligible rows before duplicate pair aggregation:", len(tier12_edge_candidates))
print("Final selected edges:", tier12_graph.number_of_edges())
print("Duplicate pair rows collapsed:", len(tier12_duplicate_edges_qa))
print("Clusters detected:", len(tier12_communities))
print("Node centrality Excel rows:", len(tier12_node_centrality))
print("Top 25% absolute strength rows:", len(tier12_top25_abs_strength_by_cluster))
print("Top 25% bridge strength rows:", len(tier12_top25_bridge_strength_by_cluster))
print("Top 25% within-cluster strength rows:", len(tier12_top25_within_strength_by_cluster))
print("Manual related nodes found:", len(manual_related_found), "of", len(TIER12_MANUAL_NODE_KEYS))
if manual_related_missing:
    print("Manual related nodes missing from related_questionnaire column:")
    for label in manual_related_missing:
        print("  -", label)
print("\nExcluded rows by reason:")
reason_counts = (
    tier12_excluded_rows_qa["exclusion_reason"]
    .str.get_dummies(sep="; ")
    .sum()
    .sort_values(ascending=False)
)
print(reason_counts.to_string())

summary_tier12 = pd.DataFrame([
    {"metric": "total_network_nodes", "value": tier12_graph.number_of_nodes()},
    {"metric": "connected_nodes", "value": len(connected_tier12_nodes)},
    {"metric": "isolated_nodes", "value": len(tier12_isolated_nodes)},
    {"metric": "positive_edges", "value": int((cleaned_tier12_edges["relation_value"] > 0).sum())},
    {"metric": "negative_edges", "value": int((cleaned_tier12_edges["relation_value"] < 0).sum())},
    {"metric": "clusters_detected", "value": len(tier12_communities)},
    {"metric": "duplicate_pair_rows_collapsed", "value": len(tier12_duplicate_edges_qa)},
])
source_type_counts = (
    cleaned_tier12_edges["relation_source_type"]
    .value_counts()
    .rename_axis("relation_source_type")
    .reset_index(name="value")
)
summary_tier12 = pd.concat([
    summary_tier12,
    source_type_counts.assign(metric=lambda table: "edges_" + table["relation_source_type"])[["metric", "value"]],
], ignore_index=True)
show_table(summary_tier12, "\nTier 1+2 network summary")
show_table(
    tier12_node_centrality.head(15),
    "\nTop 15 Tier 1+2 nodes by absolute weighted node strength",
)

# Assertions matching the implementation plan.
requested_keys = set(TIER12_MANUAL_NODE_KEYS)
node_keys = set(tier12_nodes["node_key"])
assert requested_keys.issubset(node_keys), "Not all requested related-only nodes are in the Tier 1+2 node list."
assert not cleaned_tier12_edges["relation_source_type"].isin(TIER12_EXCLUDED_SOURCE_TYPES).any(), "Excluded source type survived in cleaned edges."
selected_original_rows = set(cleaned_tier12_edges["original_row"])
selected_significance = tier12_df[tier12_df["original_row"].isin(selected_original_rows)]["significance_clean"]
assert not selected_significance.eq("no").any(), "Explicit no-significance row survived in selected edges."
unclear_selected = tier12_df[
    tier12_df["original_row"].isin(selected_original_rows)
    & tier12_df["significance_is_unclear"]
]
assert unclear_selected["relation_abs"].ge(0.02).all(), "Unclear significance selected below abs(r)=.02."
if len(tier12_duplicate_edges_qa):
    assert (
        tier12_duplicate_edges_qa["selected_relation_abs"] >= tier12_duplicate_edges_qa["relation_abs"]
    ).all(), "A duplicate pair did not keep the largest absolute relation value."

# Publication-quality Tier 1+2 figures.
TIER12_CLUSTER_PALETTE = [
    # Cluster rim colors intentionally avoid the positive/negative edge colors
    # (#D55E00 orange and #0072B2 blue), so node clusters cannot be confused
    # with edge valence or correlation labels.
    "#009E73", "#CC79A7", "#6A3D9A", "#8C564B", "#66A61E", "#E7298A",
    "#7F7F7F", "#B3A000", "#984EA3", "#4DAF4A", "#AA4499", "#117733",
]


def tier12_cluster_color(cluster_id):
    if pd.isna(cluster_id):
        return "#A8A8A8"
    return TIER12_CLUSTER_PALETTE[(int(cluster_id) - 1) % len(TIER12_CLUSTER_PALETTE)]


def normalize_layout_positions(raw_pos, scale=1.0):
    if not raw_pos:
        return {}
    xs = [xy[0] for xy in raw_pos.values()]
    ys = [xy[1] for xy in raw_pos.values()]
    center_x = (min(xs) + max(xs)) / 2
    center_y = (min(ys) + max(ys)) / 2
    span = max(max(xs) - min(xs), max(ys) - min(ys), 1e-6)
    return {
        node: ((x - center_x) / span * 2.0 * scale, (y - center_y) / span * 2.0 * scale)
        for node, (x, y) in raw_pos.items()
    }


TIER12_MANUAL_CLUSTER_LAYOUTS = {
    frozenset(["IOS", "MPS", "NADA-T", "RISC", "S-SCS-IND", "SAS", "SCS", "SFoRC"]): {
        "IOS": (-1.40, 0.92),
        "SFoRC": (1.25, 0.92),
        "NADA-T": (-0.86, 0.08),
        "MPS": (0.88, 0.08),
        "RISC": (-1.35, -0.90),
        "S-SCS-IND": (1.30, -0.82),
        "SCS": (-0.42, -1.58),
        "SAS": (0.95, -1.58),
    },
}


def spring_positions_for_nodes(graph, nodes, seed_offset=0, scale=1.0, k_multiplier=2.35):
    subgraph = graph.subgraph(nodes).copy()
    if subgraph.number_of_nodes() == 1:
        return {next(iter(subgraph.nodes)): (0.0, 0.0)}
    raw_pos = nx.spring_layout(
        subgraph,
        seed=TIER12_LAYOUT_SEED + seed_offset,
        iterations=1800,
        k=max(0.65, k_multiplier / math.sqrt(subgraph.number_of_nodes())),
        scale=1.0,
        weight=None,
    )
    return normalize_layout_positions(raw_pos, scale=scale)


def tier12_cluster_positions(graph, nodes, seed_offset=0, scale=1.0):
    short_to_node = {graph.nodes[node]["short_label"]: node for node in nodes}
    short_labels = frozenset(short_to_node)
    for required_labels, manual_layout in TIER12_MANUAL_CLUSTER_LAYOUTS.items():
        if required_labels.issubset(short_labels):
            raw_pos = {
                short_to_node[short_label]: xy
                for short_label, xy in manual_layout.items()
                if short_label in short_to_node
            }
            return normalize_layout_positions(raw_pos, scale=scale)
    return spring_positions_for_nodes(graph, nodes, seed_offset=seed_offset, scale=scale, k_multiplier=2.75)


def spread_positions(pos, min_distance=0.62, iterations=220, strength=0.34):
    """Deterministically push nearby nodes apart after the force layout.

    This is intentionally applied only to the large main component; the small
    unrelated components already have readable hand-sized layouts.
    """
    adjusted = {node: [float(x), float(y)] for node, (x, y) in pos.items()}
    nodes = list(adjusted)
    for _ in range(iterations):
        total_move = 0.0
        for i, node_a in enumerate(nodes):
            ax_, ay_ = adjusted[node_a]
            for node_b in nodes[i + 1:]:
                bx_, by_ = adjusted[node_b]
                dx = bx_ - ax_
                dy = by_ - ay_
                distance = math.hypot(dx, dy)
                if distance >= min_distance:
                    continue
                if distance < 1e-6:
                    angle = ((i + 1) * 2.399963229728653) % (2 * math.pi)
                    ux, uy = math.cos(angle), math.sin(angle)
                    distance = 1e-6
                else:
                    ux, uy = dx / distance, dy / distance
                push = (min_distance - distance) * strength
                adjusted[node_a][0] -= ux * push
                adjusted[node_a][1] -= uy * push
                adjusted[node_b][0] += ux * push
                adjusted[node_b][1] += uy * push
                total_move += abs(push)
        if total_move < 1e-4:
            break
    return {node: tuple(xy) for node, xy in adjusted.items()}


def tier12_connected_component_layout(graph, main_component_only=False):
    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    if not components:
        return {}
    if main_component_only:
        components = components[:1]

    positions = {}
    main_nodes = set(components[0])
    main_scale = 22.0 if main_component_only else 16.8
    main_pos = spring_positions_for_nodes(
        graph,
        main_nodes,
        seed_offset=700,
        scale=main_scale,
        k_multiplier=8.2 if main_component_only else 6.5,
    )
    main_pos = spread_positions(
        main_pos,
        min_distance=1.34 if main_component_only else 1.04,
        iterations=620 if main_component_only else 420,
        strength=0.54 if main_component_only else 0.44,
    )
    positions.update(main_pos)

    if main_component_only or len(components) == 1:
        return positions

    xs = [xy[0] for xy in main_pos.values()]
    ys = [xy[1] for xy in main_pos.values()]
    main_min_x, main_max_x = min(xs), max(xs)
    main_min_y, main_max_y = min(ys), max(ys)
    right_x = main_max_x + 3.4
    small_components = components[1:]
    right_count = min(3, len(small_components))
    if right_count == 1:
        right_ys = [(main_min_y + main_max_y) / 2]
    else:
        right_ys = [
            main_max_y - 0.9 - i * ((main_max_y - main_min_y - 1.8) / max(right_count - 1, 1))
            for i in range(right_count)
        ]

    for i, nodes in enumerate(small_components):
        local_scale = max(1.95, 0.86 * math.sqrt(len(nodes)))
        local_pos = tier12_cluster_positions(graph, nodes, seed_offset=900 + i, scale=local_scale)
        if i < right_count:
            center_x, center_y = right_x, right_ys[i]
        else:
            remaining_index = i - right_count
            remaining_total = len(small_components) - right_count
            if remaining_total == 1:
                center_x = (main_min_x + main_max_x) / 2
            else:
                center_x = main_min_x + (remaining_index + 1) * ((main_max_x - main_min_x) / (remaining_total + 1))
            center_y = main_min_y - 3.2
        for node, (x, y) in local_pos.items():
            positions[node] = (x + center_x, y + center_y)
    return positions


def tier12_edge_label_position(pos, source, target, graph_center=(0.0, 0.0), offset=TIER12_EDGE_LABEL_DISTANCE):
    x1, y1 = pos[source]
    x2, y2 = pos[target]
    midpoint_x = (x1 + x2) / 2
    midpoint_y = (y1 + y2) / 2
    dx = x2 - x1
    dy = y2 - y1
    length = math.hypot(dx, dy) or 1.0
    normal_x = -dy / length
    normal_y = dx / length
    away_x = midpoint_x - graph_center[0]
    away_y = midpoint_y - graph_center[1]
    if normal_x * away_x + normal_y * away_y < 0:
        normal_x *= -1
        normal_y *= -1
    return midpoint_x + normal_x * offset, midpoint_y + normal_y * offset


def set_axis_to_positions(ax, pos, pad=1.0):
    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    if not xs or not ys:
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        return
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)


def draw_tier12_network_panel(
    ax,
    graph,
    pos,
    title,
    label_width=27,
    node_font_size=5.0,
    edge_font_size=4.7,
    legend=False,
):
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.axis("off")
    set_axis_to_positions(ax, pos, pad=1.1)
    if not pos:
        return
    graph_center = (
        sum(x for x, _ in pos.values()) / len(pos),
        sum(y for _, y in pos.values()) / len(pos),
    )

    source_types_in_graph = sorted({data["relation_source_type"] for _, _, data in graph.edges(data=True)})
    for source_type in source_types_in_graph:
        edgelist = [(u, v) for u, v, data in graph.edges(data=True) if data["relation_source_type"] == source_type]
        if not edgelist:
            continue
        widths = [0.55 + 5.2 * graph.edges[edge]["abs_relation_value"] for edge in edgelist]
        colors = [graph.edges[edge]["color"] for edge in edgelist]
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=edgelist,
            edge_color=colors,
            width=widths,
            style=tier12_edge_linestyle(source_type),
            alpha=0.78,
            ax=ax,
        )

    for source, target, data in graph.edges(data=True):
        lx, ly = tier12_edge_label_position(pos, source, target, graph_center=graph_center)
        ax.text(
            lx,
            ly,
            data["edge_label"],
            color=data["color"],
            fontsize=edge_font_size,
            fontweight="bold",
            ha="center",
            va="center",
            bbox={"boxstyle": "round,pad=0.08", "facecolor": "white", "edgecolor": "none", "alpha": 0.88},
            zorder=22,
        )

    # Draw text nodes last. Square boxes = Tier 1; rounded/circular boxes = Tier 2.
    for node, data in graph.nodes(data=True):
        x, y = pos[node]
        cluster_id = data.get("cluster_id", pd.NA)
        rim_color = tier12_cluster_color(cluster_id)
        if int(data.get("tier", 2)) == 1:
            boxstyle = "square,pad=0.25"
        else:
            boxstyle = "round,pad=0.38"
        ax.text(
            x,
            y,
            plot_node_label(data["label"], width=label_width),
            ha="center",
            va="center",
            fontsize=node_font_size,
            fontweight="bold",
            linespacing=0.92,
            bbox={
                "boxstyle": boxstyle,
                "facecolor": NODE_FACE,
                "edgecolor": rim_color,
                "linewidth": 2.35,
                "alpha": 0.98,
            },
            zorder=15,
        )

    if legend:
        observed_source_types = source_types_in_graph
        legend_items = [
            Line2D([0], [0], color=POSITIVE_COLOR, lw=2.4, label="Positive relation"),
            Line2D([0], [0], color=NEGATIVE_COLOR, lw=2.4, label="Negative relation"),
            Line2D([0], [0], marker="s", color="#333333", markerfacecolor=NODE_FACE, markersize=8, lw=0, label="Tier 1 node"),
            Line2D([0], [0], marker="o", color="#333333", markerfacecolor=NODE_FACE, markersize=8, lw=0, label="Tier 2 node"),
        ]
        legend_items.extend(
            Line2D([0], [0], color="#555555", lw=2, linestyle=tier12_edge_linestyle(source_type), label=source_type_display(source_type))
            for source_type in observed_source_types
        )
        ax.legend(
            handles=legend_items,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.055),
            ncol=min(7, len(legend_items)),
            frameon=False,
            fontsize=7,
        )


def save_tier12_figure(fig, paths, dpi=300):
    fig.savefig(paths["png"], dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(paths["pdf"], bbox_inches="tight", facecolor="white")
    fig.savefig(paths["svg"], bbox_inches="tight", facecolor="white")
    for fmt, path in paths.items():
        print(f"Saved Tier 1+2 {fmt.upper()} figure: {path}")

# Remove stale cluster figure files/folders so skipped small clusters do not linger after reruns.
for old_cluster_output in OUTPUT_FIGURES_DIR.glob("tier_1_2_cluster_*"):
    if old_cluster_output.is_dir():
        shutil.rmtree(old_cluster_output)
    elif old_cluster_output.is_file():
        old_cluster_output.unlink()

# Combined connected-node figure. Isolated nodes are intentionally table-only.
tier12_combined_pos = tier12_connected_component_layout(tier12_connected_graph, main_component_only=False)
fig_tier12, ax_tier12 = plt.subplots(figsize=(56, 40))
fig_tier12.patch.set_facecolor("white")
fig_tier12.suptitle(TIER12_GRAPH_TITLE, x=0.015, y=0.99, ha="left", va="top", fontweight="bold", fontsize=15)
draw_tier12_network_panel(
    ax_tier12,
    tier12_connected_graph,
    tier12_combined_pos,
    "Weighted Louvain clusters (connected nodes only)",
    label_width=23,
    node_font_size=4.1,
    edge_font_size=4.0,
    legend=True,
)
fig_tier12.text(
    0.015,
    0.012,
    "Edges include selected significant/eligible relation values only; duplicate questionnaire pairs keep the largest absolute r. Node rim color marks Louvain cluster.",
    fontsize=7,
    ha="left",
)
save_tier12_figure(fig_tier12, TIER12_COMBINED_GRAPH_PATHS, dpi=300)
plt.close(fig_tier12)

# Main connected component only. This removes the unrelated small components and gives
# the largest network more room for readable node and edge labels.
tier12_main_component_nodes = set(max(nx.connected_components(tier12_connected_graph), key=len))
tier12_main_network_graph = tier12_connected_graph.subgraph(tier12_main_component_nodes).copy()
tier12_main_network_pos = tier12_connected_component_layout(tier12_main_network_graph, main_component_only=True)
fig_tier12_main, ax_tier12_main = plt.subplots(figsize=(68, 48))
fig_tier12_main.patch.set_facecolor("white")
fig_tier12_main.suptitle(
    "Tier 1 and 2 self questionnaires mapping network - main connected component",
    x=0.015,
    y=0.99,
    ha="left",
    va="top",
    fontweight="bold",
    fontsize=15,
)
draw_tier12_network_panel(
    ax_tier12_main,
    tier12_main_network_graph,
    tier12_main_network_pos,
    "Main connected component only",
    label_width=23,
    node_font_size=4.1,
    edge_font_size=3.65,
    legend=True,
)
fig_tier12_main.text(
    0.015,
    0.012,
    "Edges include selected significant/eligible relation values only; duplicate questionnaire pairs keep the largest absolute r. Node rim color marks Louvain cluster.",
    fontsize=7,
    ha="left",
)
save_tier12_figure(fig_tier12_main, TIER12_MAIN_NETWORK_GRAPH_PATHS, dpi=300)
plt.close(fig_tier12_main)

# One separate figure per connected Louvain cluster with at least two nodes.
tier12_cluster_figure_paths = []
for cluster_record in tier12_cluster_records:
    cluster_id = cluster_record["cluster_id"]
    cluster_nodes = [node for node, assigned_cluster in tier12_cluster_by_node.items() if assigned_cluster == cluster_id]
    if len(cluster_nodes) < TIER12_MIN_CLUSTER_FIGURE_NODES:
        continue
    cluster_graph = tier12_connected_graph.subgraph(cluster_nodes).copy()
    cluster_slug = cluster_record["cluster_slug"]
    cluster_folder_name = f"tier_1_2_cluster_{cluster_id:02d}_{cluster_slug}"
    cluster_folder = OUTPUT_FIGURES_DIR / cluster_folder_name
    cluster_folder.mkdir(parents=True, exist_ok=True)
    cluster_paths = {
        "png": cluster_folder / f"{cluster_folder_name}.png",
        "pdf": cluster_folder / f"{cluster_folder_name}.pdf",
        "svg": cluster_folder / f"{cluster_folder_name}.svg",
    }
    cluster_pos = tier12_cluster_positions(cluster_graph, cluster_graph.nodes, seed_offset=100 + cluster_id, scale=max(2.65, 0.92 * math.sqrt(len(cluster_nodes))))
    fig_width = min(20.0, max(9.6, 4.2 + 0.90 * len(cluster_nodes)))
    fig_height = min(16.0, max(7.6, 3.8 + 0.72 * len(cluster_nodes)))
    fig_cluster, ax_cluster = plt.subplots(figsize=(fig_width, fig_height))
    fig_cluster.patch.set_facecolor("white")
    draw_tier12_network_panel(
        ax_cluster,
        cluster_graph,
        cluster_pos,
        cluster_record["cluster_name"],
        label_width=27,
        node_font_size=5.8 if len(cluster_nodes) <= 10 else 5.1,
        edge_font_size=5.3 if len(cluster_nodes) <= 10 else 4.7,
        legend=True,
    )
    save_tier12_figure(fig_cluster, cluster_paths, dpi=300)
    plt.close(fig_cluster)
    tier12_cluster_figure_paths.append({
        "cluster_id": cluster_id,
        "cluster_name": cluster_record["cluster_name"],
        **{fmt: str(path) for fmt, path in cluster_paths.items()},
    })

tier12_cluster_figure_paths = pd.DataFrame(tier12_cluster_figure_paths)
if len(tier12_cluster_figure_paths):
    show_table(tier12_cluster_figure_paths, "\nTier 1+2 cluster figure files")


Tier 1+2 workbook: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/data/database_validity_extraction.xlsx
Available sheets: ['validity_edges', 'source_status']
Selected sheet: validity_edges
Actual Tier 1+2 column names:
  1. 'focal_questionnaire'
  2. 'Tier'
  3. 'is_adapted_version'
  4. 'adapted_from'
  5. 'related_questionnaire'
  6. 'is_related_questionnaire_a_subscale'
  7. 'relation_value'
  8. 'Significance'
  9. 'relation_source_type'
  10. 'article_reference'
  11. 'notes'

Tier 1+2 column mapping used:
  'focal_questionnaire' -> 'focal_questionnaire'
  'Tier' -> 'Tier'
  'related_questionnaire' -> 'related_questionnaire'
  'relation_value' -> 'relation_value'
  'Significance' -> 'significance'
  'relation_source_type' -> 'relation_source_type'



Tier 1+2 QA checks
Total rows loaded: 1337
Tier 1 rows: 286
Tier 2 rows: 1051
Network nodes total: 137
Connected nodes: 112
Isolated nodes: 25
Eligible rows before duplicate pair aggregation: 207
Final selected edges: 200
Duplicate pair rows collapsed: 7
Clusters detected: 12
Node centrality Excel rows: 137
Top 25% absolute strength rows: 32
Top 25% bridge strength rows: 32
Top 25% within-cluster strength rows: 32
Manual related nodes found: 21 of 21

Excluded rows by reason:
related_not_in_node_universe                  724
explicit_not_significant                      338
excluded_or_missing_relation_source_type      146
duplicate_pair_lower_abs_relation               7
self_loop                                       2
unclear_significance_below_abs_.02              2
unparseable_or_out_of_range_relation_value      1

Tier 1+2 network summary


,metric,value
0,total_network_nodes,137
1,connected_nodes,112
2,isolated_nodes,25
3,positive_edges,158
4,negative_edges,42
5,clusters_detected,12
6,duplicate_pair_rows_collapsed,7
7,edges_convergent_validity,109
8,edges_correlation_only,90
9,edges_regression_semipartial,1



Top 15 Tier 1+2 nodes by absolute weighted node strength


,node_key,questionnaire,short_label,tier,node_origin,cluster_id,cluster_name,cluster_slug,is_connected,degree,...,negative_strength_abs,bridge_degree,bridge_strength_abs,within_cluster_degree,within_cluster_strength_abs,bridge_strength_share,overall_absolute_strength_rank,cluster_absolute_strength_rank,cluster_bridge_strength_rank,cluster_within_strength_rank
0,(aaq-ii): acceptance and action questionnaire-ii,(AAQ-II): Acceptance and Action Questionnaire-II,AAQ-II,2,focal_questionnaire,4,"Cluster 04: AAQ-II, PHLMS-ACC, AAQ-I, TMS-D",aaq_ii_phlms_acc_aaq_i,True,9,...,1.36,2,0.36,7,4.52,0.073770,1,1,8,1
1,(scc): self-concept clarity scale,(SCC): Self-Concept Clarity Scale,SCC,1,focal_questionnaire,3,"Cluster 03: SCC, MAAS, CAMS-R, AI",scc_maas_cams_r,True,13,...,2.18,9,2.54,4,2.21,0.534737,2,1,1,5
2,(maas): mindful attention awareness scale,(MAAS): Mindful Attention Awareness Scale,MAAS,2,focal_questionnaire,3,"Cluster 03: SCC, MAAS, CAMS-R, AI",scc_maas_cams_r,True,11,...,1.05,6,2.41,5,2.20,0.522777,3,2,2,6
3,(scs - prsc): private self-consciousness scale,(SCS - PRSC): Private Self-Consciousness Scale,SCS-PRSC,2,focal_questionnaire,2,"Cluster 02: SCS-PRSC, NCS, RRQ-RE, RRQ-RU",scs_prsc_ncs_rrq_re,True,13,...,0.23,4,0.98,9,3.56,0.215859,4,1,5,1
4,(cams-r): cognitive and affective mindfulness ...,(CAMS-R): Cognitive and Affective Mindfulness ...,CAMS-R,2,focal_questionnaire,3,"Cluster 03: SCC, MAAS, CAMS-R, AI",scc_maas_cams_r,True,8,...,1.02,1,0.52,7,3.70,0.123223,5,3,4,1
5,(toa): tolerance of ambiguity scale (macdonald...,(TOA): Tolerance of Ambiguity Scale (MacDonald...,TOA,2,focal_questionnaire,1,"Cluster 01: NFCC, PFC, PNS, TOA",nfcc_pfc_pns,True,7,...,0.83,0,0.00,7,3.83,0.000000,6,1,6,1
6,(aaq-i): acceptance and action questionnaire-i,(AAQ-I): Acceptance and Action Questionnaire-I,AAQ-I,2,focal_questionnaire,4,"Cluster 04: AAQ-II, PHLMS-ACC, AAQ-I, TMS-D",aaq_ii_phlms_acc_aaq_i,True,6,...,1.46,1,0.52,5,3.22,0.139037,7,2,5,2
7,(rrq - re): rumination-reflection questionnair...,(RRQ - RE): Rumination-Reflection Questionnair...,RRQ-RE,2,focal_questionnaire,2,"Cluster 02: SCS-PRSC, NCS, RRQ-RE, RRQ-RU",scs_prsc_ncs_rrq_re,True,10,...,0.28,4,1.06,6,2.59,0.290411,8,2,3,2
8,(mpfi - in): multidimensional psychological fl...,(MPFI - IN): Multidimensional Psychological Fl...,MPFI-IN,2,focal_questionnaire,4,"Cluster 04: AAQ-II, PHLMS-ACC, AAQ-I, TMS-D",aaq_ii_phlms_acc_aaq_i,True,5,...,1.20,1,0.76,4,2.87,0.209366,9,3,2,3
9,(tms - d): toronto mindfulness scale - decente...,(TMS - D): Toronto Mindfulness Scale - Decente...,TMS-D,2,focal_questionnaire,4,"Cluster 04: AAQ-II, PHLMS-ACC, AAQ-I, TMS-D",aaq_ii_phlms_acc_aaq_i,True,6,...,0.44,3,1.30,3,1.97,0.397554,10,4,1,8


Saved Tier 1+2 PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.png
Saved Tier 1+2 PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.pdf
Saved Tier 1+2 SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.svg


... (7 outputs left out)

Saved Tier 1+2 PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_07_nada_t_mps_sforc/tier_1_2_cluster_07_nada_t_mps_sforc.png
Saved Tier 1+2 PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_07_nada_t_mps_sforc/tier_1_2_cluster_07_nada_t_mps_sforc.pdf
Saved Tier 1+2 SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_07_nada_t_mps_sforc/tier_1_2_cluster_07_nada_t_mps_sforc.svg


Saved Tier 1+2 PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c.png
Saved Tier 1+2 PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c.pdf
Saved Tier 1+2 SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c/tier_1_2_cluster_08_s_scs_int_mqi_a_mqi_c.svg


Saved Tier 1+2 PNG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_09_scd_sd_scd_var_scd_r/tier_1_2_cluster_09_scd_sd_scd_var_scd_r.png
Saved Tier 1+2 PDF figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_09_scd_sd_scd_var_scd_r/tier_1_2_cluster_09_scd_sd_scd_var_scd_r.pdf
Saved Tier 1+2 SVG figure: /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_cluster_09_scd_sd_scd_var_scd_r/tier_1_2_cluster_09_scd_sd_scd_var_scd_r.svg

Tier 1+2 cluster figure files


,cluster_id,cluster_name,png,pdf,svg
0,1,"Cluster 01: NFCC, PFC, PNS, TOA",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
1,2,"Cluster 02: SCS-PRSC, NCS, RRQ-RE, RRQ-RU",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
2,3,"Cluster 03: SCC, MAAS, CAMS-R, AI",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
3,4,"Cluster 04: AAQ-II, PHLMS-ACC, AAQ-I, TMS-D",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
4,5,"Cluster 05: IPT, SEB, BTS, IHS",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
5,6,"Cluster 06: FSC, PSCQ, PSC-N, PSC-S",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
6,7,"Cluster 07: NADA-T, MPS, SFoRC, IOS",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
7,8,"Cluster 08: S-SCS-INT, MQI-A, MQI-C, MQI-SE",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...
8,9,"Cluster 09: SCD-SD, SCD-VAR, SCD-R, SCD-CSD",/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...,/Users/dotanshani/Desktop/neuroscience Ms.c/Se...


## 9. Tier 1+2 Output Review

Confirm that the Tier 1+2 graph, cluster figures, cleaned edge list, node list, centrality Excel workbooks, cluster membership table, isolated-node table, and QA tables were written successfully.


In [9]:
tier12_expected_paths = [
    *TIER12_COMBINED_GRAPH_PATHS.values(),
    *TIER12_MAIN_NETWORK_GRAPH_PATHS.values(),
    TIER12_CLEANED_EDGES_PATH,
    TIER12_DUPLICATE_EDGES_QA_PATH,
    TIER12_NODE_LIST_PATH,
    TIER12_CLUSTER_MEMBERSHIP_PATH,
    TIER12_ISOLATED_NODES_PATH,
    TIER12_EXCLUDED_ROWS_QA_PATH,
    TIER12_EXCLUDED_NON_NETWORK_PATH,
    TIER12_NODE_CENTRALITY_XLSX_PATH,
    TIER12_TOP25_ABS_STRENGTH_XLSX_PATH,
    TIER12_TOP25_BRIDGE_STRENGTH_XLSX_PATH,
    TIER12_TOP25_WITHIN_STRENGTH_XLSX_PATH,
]
if "tier12_cluster_figure_paths" in globals() and len(tier12_cluster_figure_paths):
    for _, row in tier12_cluster_figure_paths.iterrows():
        tier12_expected_paths.extend([Path(row["png"]), Path(row["pdf"]), Path(row["svg"])])

missing_tier12_outputs = [path for path in tier12_expected_paths if not Path(path).exists()]
if missing_tier12_outputs:
    raise FileNotFoundError("Missing expected Tier 1+2 outputs:\n" + "\n".join(map(str, missing_tier12_outputs)))

print("Tier 1+2 output files created:")
for path in tier12_expected_paths:
    print(f"  - {path}")


Tier 1+2 output files created:
  - /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.png
  - /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.pdf
  - /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_questionnaires_mapping_network.svg
  - /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_main_network_questionnaires_mapping_network.png
  - /Users/dotanshani/Desktop/neuroscience Ms.c/Self-Reification Questionnaire Mapping Project/questionnaire_mapping_network_analysis/outputs/figures/tier_1_2_main_network_question